# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaryumAkram16/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal Check 1 — Staleness (behind the `stale_visible_page` refresh flag): MIXED.**
Decline rate does not rise monotonically with staleness (51.2% → 61.1% → 46.7% → 60.0% across increasing staleness buckets). The two oldest buckets also have very small n (169 and 5 rows), so the apparent uptick at 365d+ is unreliable. Staleness alone is not a clean predictor of decline in this data — a useful negative result that stops me from over-weighting it.

**Signal Check 2 — CTR vs. Position (behind the `low_ctr_visible_page` CTR-fix flag): CONFIRMED.**
CTR falls steadily and substantially by position tier: 1.484 (top_3) → 0.652 (page_1) → 0.323 (striking) → 0.222 (page_3_5) → 0.150 (deep). This is a strong, monotonic signal — exactly what backs FlyRank's CTR-fix logic.

**My rule (revised in light of the checks):** Because staleness alone was a mixed signal, I lower its weight rather than treat it as an equal partner to CTR. The rule flags a page when it shows a real CTR gap for its position tier, with staleness as a secondary tiebreaker rather than a primary driver.

**Score:** `baseline_action_score = 0.7 * ctr_gap_score + 0.3 * staleness_score`

**Reason code (single):** `underperforming_ctr_and_stale`

**Action label:** `review_for_refresh` if score is in the top quartile, else `monitor`

**Volume floor:** Only pages with `impressions_90d >= 500` are scored at all (matching the real `low_ctr_visible_page` flag's own threshold). Early testing without this floor showed the top-20 dominated by pages with 1-4 impressions and a mechanical 0% CTR — noise, not signal — so this floor is a required part of the rule, not an optional refinement.


In [11]:
import pandas as pd

url = "https://raw.githubusercontent.com/MaryumAkram16/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# --- Signal Check 1: Staleness (behind stale_visible_page flag) ---
df["staleness_tier"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 89, 179, 364, 10_000],
    labels=["<90d", "90-179d", "180-364d", "365d+"]
)
staleness_check = df.groupby("staleness_tier").agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda s: (s == "down").mean())
)
print("Signal Check 1 — Staleness vs. decline rate:")
print(staleness_check)
print()

# --- Signal Check 2: CTR vs. Position (behind low_ctr_visible_page flag) ---
ctr_check = df.groupby("position_tier").agg(
    n=("content_id", "count"),
    avg_ctr=("ctr", "mean")
).sort_values("avg_ctr", ascending=False)
print("Signal Check 2 — CTR by position tier:")
print(ctr_check)


Signal Check 1 — Staleness vs. decline rate:
                    n  decline_rate
staleness_tier                     
<90d            20655      0.512031
90-179d          9171      0.611057
180-364d          169      0.467456
365d+               5      0.600000

Signal Check 2 — CTR by position tier:
                   n   avg_ctr
position_tier                 
top_3           2321  1.483611
page_1         11814  0.652467
striking        7304  0.323239
page_3_5        7242  0.222484
deep            1319  0.150212


/tmp/ipykernel_3980/3325530792.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  staleness_check = df.groupby("staleness_tier").agg(


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
import numpy as np
import os

# --- Minimum volume floor: exclude pages too small to trust a CTR measurement ---
# (matches the real low_ctr_visible_page flag's own threshold)
MIN_IMPRESSIONS = 500
eligible = df[df["impressions_90d"] >= MIN_IMPRESSIONS].copy()
excluded_count = len(df) - len(eligible)
print(f"Excluded {excluded_count} of {len(df)} pages below the {MIN_IMPRESSIONS}-impression floor "
      f"(too little traffic to trust a CTR measurement).")

# --- Normalize component scores to 0-1, computed only among eligible pages ---
eligible["staleness_score"] = (eligible["days_since_last_update"] / eligible["days_since_last_update"].max()).clip(0, 1)

expected_ctr_by_tier = eligible.groupby("position_tier")["ctr"].transform("mean")
eligible["ctr_gap_raw"] = (expected_ctr_by_tier - eligible["ctr"]).clip(lower=0)
eligible["ctr_gap_score"] = (eligible["ctr_gap_raw"] / eligible["ctr_gap_raw"].max()).clip(0, 1)

# --- Combine into the baseline score ---
eligible["baseline_action_score"] = 0.7 * eligible["ctr_gap_score"] + 0.3 * eligible["staleness_score"]

# --- One reason code, one action label ---
threshold = eligible["baseline_action_score"].quantile(0.75)
eligible["reason_code"] = "underperforming_ctr_and_stale"
eligible["action"] = np.where(eligible["baseline_action_score"] >= threshold, "review_for_refresh", "monitor")

# --- Rank and write the CSV ---
ranked = eligible.sort_values("baseline_action_score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)
output_cols = ["content_id", "client_id", "baseline_action_score", "reason_code", "action",
               "days_since_last_update", "position_tier", "ctr", "impressions_90d"]
ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(ranked)} ranked rows to work/outputs/baseline_action_score.csv")
print(f"\nTop 20 preview:")
ranked[output_cols].head(20)

Excluded 13274 of 30000 pages below the 500-impression floor (too little traffic to trust a CTR measurement).
Wrote 16726 ranked rows to work/outputs/baseline_action_score.csv

Top 20 preview:


,content_id,client_id,baseline_action_score,reason_code,action,days_since_last_update,position_tier,ctr,impressions_90d
28081,content_a37ce8ddc090,client_7f2253d7e2,0.805648,underperforming_ctr_and_stale,review_for_refresh,106,top_3,0.0,1039
29354,content_1f7638fecdc1,client_3fdba35f04,0.803654,underperforming_ctr_and_stale,review_for_refresh,104,top_3,0.0,565
25422,content_7d5ad3f9feee,client_19581e27de,0.803654,underperforming_ctr_and_stale,review_for_refresh,104,top_3,0.0,1916
9294,content_04c10b7d4b2b,client_19581e27de,0.803654,underperforming_ctr_and_stale,review_for_refresh,104,top_3,0.0,1179
11395,content_7e97f31ba9bb,client_19581e27de,0.803654,underperforming_ctr_and_stale,review_for_refresh,104,top_3,0.0,603
24730,content_d0fd2e1ec8f6,client_6208ef0f77,0.803654,underperforming_ctr_and_stale,review_for_refresh,104,top_3,0.0,519
21822,content_1c0f8c2f9018,client_19581e27de,0.803654,underperforming_ctr_and_stale,review_for_refresh,104,top_3,0.0,615
14115,content_5cda32af644c,client_3fdba35f04,0.803654,underperforming_ctr_and_stale,review_for_refresh,104,top_3,0.0,720
18123,content_9e6c26757e7b,client_4e07408562,0.803654,underperforming_ctr_and_stale,review_for_refresh,104,top_3,0.0,1135
8468,content_26e7e98c8e41,client_6208ef0f77,0.803654,underperforming_ctr_and_stale,review_for_refresh,104,top_3,0.0,1226


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

Nearly the entire top-20 shares the same profile: `position_tier = top_3`, `ctr = 0.0`, `days_since_last_update ≈ 104-106`, `impressions_90d` ranging from ~519 to ~1986. Rather than write 20 near-identical lines, here is the pattern plus what makes it notable per row:

**Action for all 20:** `review_for_refresh` — flagged because they combine the two signals: a real CTR gap for their tier (`0.7` weight) plus moderate staleness (`0.3` weight).

**Why they're here:** Each page sits in the best possible position tier (`top_3`) with substantial traffic (500-2000 impressions over 90 days) yet reports **exactly 0% CTR**. Against the tier's real average CTR of 1.48 (from Signal Check 2), this is the maximum possible CTR gap — which is why these dominate the top of the ranked list.

**What would make each one wrong:** A `top_3` page with 500+ impressions and literally zero clicks over 90 days is unusual — real searchers rarely ignore a top-3 result completely, even with a weak title. The more likely explanation for several of these is a **tracking or data issue** (a broken GSC-to-page link, a redirect not being tracked, or a robots/canonical issue hiding the real click data) rather than a genuine content problem. Before recommending an actual content rewrite, a reviewer should first verify the page is indexed correctly and that clicks are being tracked at all — refreshing the content itself would be the wrong first action if the real issue is a broken tracking pipe.

**Individual rows** (content_id — impressions — distinguishing note):
| content_id | impressions_90d | days_since_last_update | note |
|---|---|---|---|
| content_a37ce8ddc090 | 1039 | 106 | highest score in the set — slightly more stale than the rest |
| content_1f7638fecdc1 | 565 | 104 | lowest impression volume among the tied group — most likely candidate for genuine low visibility rather than a tracking bug |
| content_7d5ad3f9feee | 1916 | 104 | highest impressions in the tied group — the zero-CTR-at-this-volume anomaly is strongest here, most worth a tracking audit first |
| content_04c10b7d4b2b | 1179 | 104 | mid-volume, same pattern |
| content_7e97f31ba9bb | 603 | 104 | near the volume floor — worth double-checking it clears 500 by a real margin, not a rounding artifact |
| (remaining 15 rows) | 519–1986 | 104 | same profile: top_3 + zero CTR + ~104 days stale — all warrant a tracking check before a content rewrite |

In [13]:
# Show the actual top-20 rows referenced in the review above, as evidence for the claims
top20 = ranked[output_cols].head(20)
print(top20.to_string(index=False))

# Confirm the pattern claim: how many of the top 20 share position_tier == 'top_3' and ctr == 0.0?
top20_top3_zero_ctr = ((top20["position_tier"] == "top_3") & (top20["ctr"] == 0.0)).sum()
print(f"\n{top20_top3_zero_ctr} of the top 20 rows are top_3 position with 0.0 CTR — confirming the pattern described above.")


          content_id         client_id  baseline_action_score                   reason_code             action  days_since_last_update position_tier  ctr  impressions_90d
content_a37ce8ddc090 client_7f2253d7e2               0.805648 underperforming_ctr_and_stale review_for_refresh                     106         top_3  0.0             1039
content_1f7638fecdc1 client_3fdba35f04               0.803654 underperforming_ctr_and_stale review_for_refresh                     104         top_3  0.0              565
content_7d5ad3f9feee client_19581e27de               0.803654 underperforming_ctr_and_stale review_for_refresh                     104         top_3  0.0             1916
content_04c10b7d4b2b client_19581e27de               0.803654 underperforming_ctr_and_stale review_for_refresh                     104         top_3  0.0             1179
content_7e97f31ba9bb client_19581e27de               0.803654 underperforming_ctr_and_stale review_for_refresh                     104         to

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** Only the single top row (`content_a37ce8ddc090`, score 0.805648) is unique — the next 24 rows are all tied at an identical score (0.803654), because those pages share matching `days_since_last_update`, `ctr`, and `position_tier` values. This tied cluster fully accounts for positions 2-20 in the review list — every one of my top-20 rows below rank 1 is drawn from this same 24-row tie. This is a real limitation of the rule: within the tie, there is no principled way to say which page should be reviewed first — the order is arbitrary (whatever pandas' sort happens to return). A stronger version of this rule would need an explicit tie-breaker, such as sorting ties by raw impressions so higher-traffic pages within a tie get reviewed first.

**Leakage check:** Confirmed below — the score uses only `days_since_last_update`, `ctr`, `position_tier`, and `impressions_90d`. None of these are label-derived or future-window fields; `trend_direction` and `trend_pct` (the fields the ML-01–03 label was built from) were never used as inputs to this rule.

In [14]:
scoring_inputs = ["days_since_last_update", "ctr", "position_tier", "impressions_90d"]
label_derived_fields = ["trend_direction", "trend_pct"]

leaked = [col for col in scoring_inputs if col in label_derived_fields]
print(f"Scoring inputs used: {scoring_inputs}")
print(f"Leakage check — any label-derived fields in scoring inputs? {leaked if leaked else 'None — clean.'}")

# The single highest score is unique; the tie cluster is the score just below it
top_score = ranked["baseline_action_score"].iloc[0]
second_score = ranked["baseline_action_score"].iloc[1]
tie_count = ranked[ranked["baseline_action_score"] == second_score].shape[0]

print(f"\nTop score ({top_score:.6f}): unique, 1 row.")
print(f"Second-place score ({second_score:.6f}): {tie_count} rows tied — this is the cluster dominating positions 2-20.")


Scoring inputs used: ['days_since_last_update', 'ctr', 'position_tier', 'impressions_90d']
Leakage check — any label-derived fields in scoring inputs? None — clean.

Top score (0.805648): unique, 1 row.
Second-place score (0.803654): 24 rows tied — this is the cluster dominating positions 2-20.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.